<a href="https://colab.research.google.com/github/pranathi139/GenAI/blob/main/Week2_Hybrid_Retrieval_Comparision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-community faiss-cpu rank-bm25


In [ ]:

import os

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi


In [ ]:
docs = [
    Document(page_content="Hybrid retrieval combines vector similarity and keyword matching."),
    Document(page_content="Vector search uses embeddings to retrieve semantically similar text."),
    Document(page_content="Keyword search relies on exact word matching like BM25."),
    Document(page_content="Hybrid retrieval improves factual accuracy in RAG systems."),
    Document(page_content="FAISS is a library for efficient vector similarity search.")
]

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)



In [ ]:
tokenized_docs = [doc.page_content.split() for doc in docs]
bm25 = BM25Okapi(tokenized_docs)

def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.split())
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return ranked[:k]

In [ ]:
def hybrid_retrieve(query, k=3, alpha=0.5):
    vector_results = vectorstore.similarity_search_with_score(query, k=k)
    bm25_results = bm25_retrieve(query, k=k)

    combined_scores = {}

    for doc, score in vector_results:
        combined_scores[doc.page_content] = alpha * (1 / (1 + score))

    for score, doc in bm25_results:
        combined_scores[doc.page_content] = combined_scores.get(
            doc.page_content, 0
        ) + (1 - alpha) * score

    return sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)


Compare Vector vs Hybrid Results

In [ ]:
query = "What is hybrid retrieval?"
print("VECTOR SEARCH RESULTS:\n")
vector_hits = vectorstore.similarity_search_with_score(query, k=3)

for doc, score in vector_hits:
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print()

In [ ]:
query = "What is hybrid retrieval?"
print("HYBRID SEARCH RESULTS:\n")
hybrid_hits = hybrid_retrieve(query, k=3)

for text, score in hybrid_hits:
    print(f"Hybrid Score: {score:.4f}")
    print(text)
    print()